# 1.Late Interaction Models

## a. COLBERT

In [ ]:
# !pip install -U einops flash_attn
!pip install -U pylate


In [ ]:
from pylate import indexes, models, retrieve
import json
import numpy as np
import os
from tqdm import tqdm
import pickle
import time

In [ ]:
colbert_model = models.ColBERT(
    model_name_or_path="jinaai/jina-colbert-v2",
    query_prefix="[QueryMarker]",
    document_prefix="[DocumentMarker]",
    attend_to_expansion_tokens=True,
    trust_remote_code=True,
)
colbert_model.query_length = 192


In [ ]:
def embed_texts(batch_texts, model,doc_type):
    if doc_type=="query":
      return model.encode(batch_texts,is_query=True)
    else:
      return model.encode(batch_texts,is_query=False)

In [ ]:
def build_colbert_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):

    all_embeddings = []
    total_time = 0

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        start=time.time()

        emb = embed_texts(
            batch,
            model,
            doc_type="document"
        )

        end=time.time()

        total_time += end-start

        print(
            f"batch {i//batch_size+1} "
            f"time={end-start:.4f}s"
        )

        all_embeddings.extend(emb)


    if save_path:

        os.makedirs(save_path,exist_ok=True)

        with open(
            os.path.join(
                save_path,
                "colbert_embeddings_docs.pkl"
            ),
            "wb"
        ) as f:
            pickle.dump(all_embeddings,f)


        np.save(
            os.path.join(save_path,"doc_ids.npy"),
            ids
        )

    print("Documents:",len(ids))

    return all_embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

colbert_embeddings= build_colbert_pipeline(
    texts=texts,
    ids=ids,
    model=colbert_model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

In [ ]:
# Embed queries using ColBERT
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"

output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/"

os.makedirs(output_dir, exist_ok=True)

embedding_file = os.path.join(output_dir, "query_embeddings.pkl")
metadata_file = os.path.join(output_dir, "query_metadata.jsonl")
ids_file = os.path.join(output_dir, "query_ids.npy")

query_embeddings = []
query_ids = []

start_time = time.time()

batch_items = []
batch_texts = []

num_queries = 0


with open(metadata_file, "w", encoding="utf-8") as meta_f:

    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)

            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)


            if len(batch_texts) == BATCH_SIZE:

                # ColBERT query embeddings
                embeddings = embed_texts(
                    batch_texts,
                    colbert_model,
                    doc_type="query"
                )

                for it, emb in zip(batch_items, embeddings):

                    query_embeddings.append(emb)

                    query_ids.append(it["query_id"])


                    metadata = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "gold": it["relevant_subarticles"]
                    }

                    meta_f.write(
                        json.dumps(
                            metadata,
                            ensure_ascii=False
                        ) + "\n"
                    )

                    num_queries += 1


                batch_items = []
                batch_texts = []


        # Remaining batch
        if batch_texts:

            embeddings = embed_texts(
                    batch_texts,
                    colbert_model,
                    doc_type="query"
                )


            for it, emb in zip(batch_items, embeddings):

                query_embeddings.append(emb)

                query_ids.append(it["query_id"])


                metadata = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "gold": it["relevant_subarticles"]
                }

                meta_f.write(
                    json.dumps(
                        metadata,
                        ensure_ascii=False
                    ) + "\n"
                )

                num_queries += 1




end_time = time.time()


# Save token embeddings
with open(embedding_file, "wb") as f:
    pickle.dump(query_embeddings, f)

# Save query IDs
np.save(ids_file, np.array(query_ids))

total_time = end_time - start_time


print("\n================ QUERY EMBEDDING STATS ================")
print(f"Total queries embedded : {num_queries}")
print(f"Total embedding time   : {total_time:.2f} sec")
print(f"Average time/query     : {total_time/num_queries:.6f} sec")

print("Embedding example shape:")
print(query_embeddings[0].shape)

print("========================================================")

Embedding queries: 5731it [05:39, 16.86it/s]



================ QUERY EMBEDDING STATS ================
Total queries embedded : 5731
Total embedding time   : 346.66 sec
Average time/query     : 0.060489 sec
Embedding example shape:
(192, 128)


In [ ]:
document_embedding_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/colbert_embeddings_docs.pkl"
documents_ids_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/doc_ids.npy"

# Load document embeddings
with open(document_embedding_file, "rb") as f:
    documents_embeddings = pickle.load(f)

# Load document IDs
documents_ids = np.load(documents_ids_file)


In [ ]:
index = indexes.PLAID(
    index_folder="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/pylate_index_document",
    index_name="index",
    override=True,
)
retriever = retrieve.ColBERT(index=index)

In [ ]:
import time

start_time = time.time()

index.add_documents(
    documents_ids=documents_ids,
    documents_embeddings=documents_embeddings
)

total_time = time.time() - start_time

print(f"Total indexing time: {total_time:.2f}s")

Total indexing time: 773.11s


In [ ]:
#run experiments
query_embedding_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/query_embeddings.pkl"
query_ids_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/query_ids.npy"
metadata_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/query_metadata.jsonl"


#  LOAD QUERY EMBEDDINGS

with open(query_embedding_file, "rb") as f:
    query_embeddings = pickle.load(f)

query_ids = np.load(query_ids_file)


#  LOAD QUERY METADATA

query_metadata = []

with open(metadata_file, "r", encoding="utf-8") as f:
    for line in f:
        query_metadata.append(json.loads(line))


gold_labels = [
    x["gold"] for x in query_metadata
]

print("Queries:", len(query_embeddings))
print("Example query shape:", query_embeddings[0].shape)

# ---------------- RETRIEVAL ----------------
top_k = 100
batch_size = 128


all_results_for_metrics = []


start_time = time.time()


for i in tqdm(
    range(0, len(query_embeddings), batch_size),
    desc="Retrieving"
):

    batch_embeddings = query_embeddings[i:i+batch_size]

    # Batch retrieval
    results = retriever.retrieve(
        queries_embeddings=batch_embeddings,
        k=top_k
    )
    for j, query_results in enumerate(results):

        retrieved_ids = [
            r["id"] for r in query_results
        ]

        all_results_for_metrics.append(
            (
                retrieved_ids,
                gold_labels[i+j]
            )
        )


end_time = time.time()


# -------- STATS --------

retrieval_time = end_time - start_time
num_queries = len(query_embeddings)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries        : {num_queries}")
print(f"Total retrieval time : {retrieval_time:.4f} sec")
print(f"Avg time/query       : {retrieval_time/num_queries:.6f} sec")
print(f"Throughput           : {num_queries/retrieval_time:.2f} queries/sec")
print("=================================================")

Queries: 5731
Example query shape: (192, 128)


Retrieving: 100%|██████████| 45/45 [14:48<00:00, 19.75s/it]


================ RETRIEVAL STATS ================
Total queries        : 5731
Total retrieval time : 888.7068 sec
Avg time/query       : 0.155070 sec
Throughput           : 6.45 queries/sec


### Efficiency

In [ ]:
#final indexing time computation
# Warmup
_ = embed_texts(
    texts[:BATCH_SIZE],
    colbert_model,
    doc_type="document"
)


# Measure indexing only
start_time = time.perf_counter()

colbert_embeddings = build_colbert_pipeline(
    texts=texts,
    ids=ids,
    model=colbert_model,
    batch_size=BATCH_SIZE,
    save_path=None   # exclude disk writing
)

end_time = time.perf_counter()


indexing_time = end_time - start_time

avg_time_per_doc = indexing_time / num_docs
throughput = num_docs / indexing_time


print("\n============== INDEXING STATS ==============")
print(f"Documents embedded : {num_docs}")
print(f"Indexing time      : {indexing_time:.2f} sec")
print(f"Time per document  : {avg_time_per_doc:.6f} sec")
print(f"Throughput         : {throughput:.2f} docs/sec")
print("============================================")

In [ ]:
from tqdm import tqdm
import time
import numpy as np

##retrieval check
def measure_search_time(query_embeddings, retriever, top_k=100, batch_size=128):

    # warm-up
    _ = retriever.retrieve(
        queries_embeddings=query_embeddings[:batch_size],
        k=top_k
    )

    times = []

    for run in range(3):

        start = time.perf_counter()

        for i in tqdm(
            range(0, len(query_embeddings), batch_size),
            desc=f"Search run {run+1}/3"
        ):

            batch_embeddings = query_embeddings[i:i+batch_size]

            _ = retriever.retrieve(
                queries_embeddings=batch_embeddings,
                k=top_k
            )

        end = time.perf_counter()

        times.append(end-start)

    return times,np.mean(times), np.std(times)

times,mean_time, std_time = measure_search_time(
    query_embeddings,
    retriever,
    top_k=100,
    batch_size=128
)

print(f"Times={times}")
print(f"Search time: {mean_time:.2f} ± {std_time:.2f} sec")
print(f"Avg/query: {mean_time/len(query_embeddings):.6f} sec")

Search run 1/3:  17%|█▋        | 8/47 [02:25<11:45, 18.08s/it]

In [ ]:
import os

def get_directory_size(folder_path):
    total_size = 0

    for root, _, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            total_size += os.path.getsize(file_path)

    size_mb = total_size / (1024 ** 2)
    size_gb = total_size / (1024 ** 3)

    print(f"Folder: {os.path.basename(folder_path)}")
    print(f"Index size: {size_mb:.2f} MB")
    print(f"Index size: {size_gb:.3f} GB")

    return size_gb


index_folder = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/colbert-jina/pylate_index_document"

colbert_index_size = get_directory_size(index_folder)

Folder: pylate_index_document
Index size: 324.96 MB
Index size: 0.317 GB


### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2256,0.0824,NaN,NaN
1,10,0.2706,0.0548,0.1823,0.1975
2,100,0.4141,0.0094,NaN,NaN


# 2.Dense Embeddings

## Monolingual

### a. Universalml/Nepali Embedding Model

In [ ]:
!pip install faiss-gpu

In [ ]:
import os
import json
import numpy as np
import time
import faiss
from tqdm import tqdm

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("universalml/Nepali_Embedding_Model")

In [ ]:
def embed_texts(batch_texts, model):
    return model.encode(batch_texts)

In [ ]:
def build_faiss_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0
    index=None
    embeddings=None

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)


    # --- SAVE ARTIFACTS ---
    if save_path:
        embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")
        dim = np.array(embeddings).shape[1]
        index = faiss.IndexFlatIP(dim)
        index = faiss.IndexIDMap(index)
        index.add_with_ids(embeddings, ids)

        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [06:34, 14.54it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/queries_embed.jsonl

Total embedding time: 401.03 seconds
Average time per query: 0.0700 sec


In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))

query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model
    )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")

In [ ]:
import time
import numpy as np


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embedding_model.encode(
            [queries[0]],
            convert_to_numpy=True
        ).astype("float32")

        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embedding_model.encode(
                [query],
                convert_to_numpy=True
            ).astype("float32")

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)


    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }


#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/universalml-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 3/3 [22:10<00:00, 443.42s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 3
Total time              : 443.41 sec
Avg latency per query   : 77.37 ms
Throughput              : 12.92 queries/sec


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })
    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2603,0.0903,NaN,NaN
1,10,0.3075,0.0601,0.1939,0.2107
2,100,0.4682,0.0106,NaN,NaN


### b. premmm/nepali-embedder-v1


In [ ]:
import warnings
warnings.filterwarnings("ignore")
!pip install --upgrade sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("premmm/nepali-embedder-v1")

In [ ]:
def embed_texts(batch_texts, model):
    """
    BGE-M3 HF embedding (dense only)
    """
    return model.encode(batch_texts)

In [ ]:
def build_faiss_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    dim = np.array(embeddings).shape[1]
    index = faiss.IndexFlatIP(dim)
    index = faiss.IndexIDMap(index)
    index.add_with_ids(embeddings, ids)

    # --- SAVE ARTIFACTS ---
    if save_path:
        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()   #

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [01:26, 66.55it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/queries_embed.jsonl

Total embedding time: 87.78 seconds
Average time per query: 0.0153 sec


In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

Loading all embeddings from: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/queries_embed.jsonl
Performing batch search for 5731 queries...

================ RETRIEVAL STATS ================
Total queries           : 5731
Total retrieval time    : 2.0332 sec
Avg time per query      : 0.000355 sec
Throughput             : 2818.64 queries/sec

Batch retrieval complete.


#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model
    )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")


In [ ]:
import time
import numpy as np
import pandas as pd


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embedding_model.encode(
            [queries[0]],
            convert_to_numpy=True
        ).astype("float32")

        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embedding_model.encode(
                [query],
                convert_to_numpy=True
            ).astype("float32")

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)

    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }


#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 3/3 [07:17<00:00, 145.95s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 3
Total time              : 145.95 sec
Avg latency per query   : 25.47 ms
Throughput              : 39.27 queries/sec


In [ ]:
import os

def get_index_size(index_path):
    """
    Returns FAISS index size in MB/GB.
    """
    size_bytes = os.path.getsize(index_path)

    return {
        "size_bytes": size_bytes,
        "size_MB": size_bytes / (1024 ** 2),
        "size_GB": size_bytes / (1024 ** 3)
    }

index_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/prem-nepali/sa_faiss.index"

index_size = get_index_size(index_file)

print(f"Index size: {index_size['size_MB']:.2f} MB")
print(f"Index size: {index_size['size_GB']:.3f} GB")

Index size: 65.58 MB
Index size: 0.064 GB


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.0955,0.0350,NaN,NaN
1,10,0.1271,0.0261,0.0702,0.082
2,100,0.2750,0.0063,NaN,NaN


### c. acostillio/SemantiSearchNepaliSbert

In [ ]:
!pip install faiss-gpu

In [ ]:

import numpy as np
import pandas as pd
import faiss
import os
import torch
from transformers import AutoTokenizer, AutoModel
import time
import json
from tqdm import tqdm
import torch.nn.functional as F

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("acostillio/SemantiSearchNepaliSbert")

In [ ]:
sum(p.numel() for p in model.parameters())

237556224

In [ ]:
def embed_texts(batch_texts, model):
    return model.encode(batch_texts)

In [ ]:
def build_faiss_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    dim = np.array(embeddings).shape[1]
    index = faiss.IndexFlatIP(dim)
    index = faiss.IndexIDMap(index)
    index.add_with_ids(embeddings, ids)

    # --- SAVE ARTIFACTS ---
    if save_path:
        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

Loading documents...
Total documents: 22326
batch 1 | size=128 | time=0.2266s
batch 2 | size=128 | time=0.2234s
batch 3 | size=128 | time=0.2373s
batch 4 | size=128 | time=0.2772s
batch 5 | size=128 | time=0.2582s
batch 6 | size=128 | time=0.2829s
batch 7 | size=128 | time=0.2690s
batch 8 | size=128 | time=0.2580s
batch 9 | size=128 | time=0.2901s
batch 10 | size=128 | time=0.2852s
batch 11 | size=128 | time=0.3042s
batch 12 | size=128 | time=0.3066s
batch 13 | size=128 | time=0.3275s
batch 14 | size=128 | time=0.3325s
batch 15 | size=128 | time=0.3419s
batch 16 | size=128 | time=0.3131s
batch 17 | size=128 | time=0.3280s
batch 18 | size=128 | time=0.3394s
batch 19 | size=128 | time=0.3482s
batch 20 | size=128 | time=0.3581s
batch 21 | size=128 | time=0.3584s
batch 22 | size=128 | time=0.3621s
batch 23 | size=128 | time=0.3717s
batch 24 | size=128 | time=0.3862s
batch 25 | size=128 | time=0.3760s
batch 26 | size=128 | time=0.3846s
batch 27 | size=128 | time=0.3911s
batch 28 | size=128 

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/citation_removed_queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/citation_removed_queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()   #

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [01:57, 48.85it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/citation_removed_queries_embed.jsonl

Total embedding time: 119.81 seconds
Average time per query: 0.0209 sec


In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))

query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/citation_removed_queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

Loading all embeddings from: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/citation_removed_queries_embed.jsonl
Performing batch search for 5731 queries...

================ RETRIEVAL STATS ================
Total queries           : 5731
Total retrieval time    : 3.6836 sec
Avg time per query      : 0.000643 sec
Throughput             : 1555.81 queries/sec

Batch retrieval complete.


#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model
    )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")



Running indexing experiment 1/3...
batch 1 | size=128 | time=0.1544s
batch 2 | size=128 | time=0.1726s
batch 3 | size=128 | time=0.1911s
batch 4 | size=128 | time=0.2209s
batch 5 | size=128 | time=0.2027s
batch 6 | size=128 | time=0.2223s
batch 7 | size=128 | time=0.2255s
batch 8 | size=128 | time=0.2236s
batch 9 | size=128 | time=0.2365s
batch 10 | size=128 | time=0.2392s
batch 11 | size=128 | time=0.2521s
batch 12 | size=128 | time=0.2421s
batch 13 | size=128 | time=0.2582s
batch 14 | size=128 | time=0.2690s
batch 15 | size=128 | time=0.2681s
batch 16 | size=128 | time=0.2559s
batch 17 | size=128 | time=0.2517s
batch 18 | size=128 | time=0.2680s
batch 19 | size=128 | time=0.2893s
batch 20 | size=128 | time=0.2900s
batch 21 | size=128 | time=0.2682s
batch 22 | size=128 | time=0.2942s
batch 23 | size=128 | time=0.3041s
batch 24 | size=128 | time=0.2898s
batch 25 | size=128 | time=0.3026s
batch 26 | size=128 | time=0.2923s
batch 27 | size=128 | time=0.3174s
batch 28 | size=128 | time=0

In [ ]:
import time
import numpy as np


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embedding_model.encode(
            [queries[0]],
            convert_to_numpy=True
        ).astype("float32")

        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embedding_model.encode(
                [query],
                convert_to_numpy=True
            ).astype("float32")

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)


    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }

#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 3/3 [10:12<00:00, 204.03s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 3
Total time              : 204.03 sec
Avg latency per query   : 35.60 ms
Throughput              : 28.09 queries/sec


In [ ]:
import os

def get_index_size(index_path):
    """
    Returns FAISS index size in MB/GB.
    """
    size_bytes = os.path.getsize(index_path)

    return {
        "size_bytes": size_bytes,
        "size_MB": size_bytes / (1024 ** 2),
        "size_GB": size_bytes / (1024 ** 3)
    }

index_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/sa_faiss.index"

index_size = get_index_size(index_file)

print(f"Index size: {index_size['size_MB']:.2f} MB")
print(f"Index size: {index_size['size_GB']:.3f} GB")

Index size: 65.58 MB
Index size: 0.064 GB


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2396,0.0882,NaN,NaN
1,10,0.3068,0.0629,0.1761,0.203
2,100,0.5494,0.0126,NaN,NaN


## Multilingual

### a. BGE-M3

In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 6.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import faiss
import os
import torch
from transformers import AutoTokenizer, AutoModel
import time
import json
from tqdm import tqdm

In [ ]:
model_name = "BAAI/bge-m3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts(batch_texts, model, tokenizer):
    """
    BGE-M3 HF embedding (dense only)
    """
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])

    # normalize for cosine similarity
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().astype("float32")

def embed_query(text):
    inputs = tokenizer(
        [text],
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    emb = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)

    return emb.cpu().numpy().astype("float32")

In [ ]:
def build_faiss_pipeline(
    texts,
    ids,
    model,
    tokenizer,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model, tokenizer)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    dim = np.array(embeddings).shape[1]
    index = faiss.IndexFlatIP(dim)
    index = faiss.IndexIDMap(index)
    index.add_with_ids(embeddings, ids)

    # --- SAVE ARTIFACTS ---
    if save_path:
        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

Loading documents...
Total documents: 22326
batch 1 | size=128 | time=0.8566s
batch 2 | size=128 | time=0.5927s
batch 3 | size=128 | time=0.6807s
batch 4 | size=128 | time=0.7725s
batch 5 | size=128 | time=0.7448s
batch 6 | size=128 | time=0.9348s
batch 7 | size=128 | time=0.9137s
batch 8 | size=128 | time=0.8404s
batch 9 | size=128 | time=0.9469s
batch 10 | size=128 | time=0.9612s
batch 11 | size=128 | time=0.8957s
batch 12 | size=128 | time=1.0125s
batch 13 | size=128 | time=1.0022s
batch 14 | size=128 | time=1.2698s
batch 15 | size=128 | time=1.0956s
batch 16 | size=128 | time=0.9933s
batch 17 | size=128 | time=0.9965s
batch 18 | size=128 | time=1.0160s
batch 19 | size=128 | time=1.1392s
batch 20 | size=128 | time=1.1401s
batch 21 | size=128 | time=1.1505s
batch 22 | size=128 | time=1.0626s
batch 23 | size=128 | time=1.2701s
batch 24 | size=128 | time=1.2048s
batch 25 | size=128 | time=1.3048s
batch 26 | size=128 | time=1.1494s
batch 27 | size=128 | time=1.3445s
batch 28 | size=128 

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model, tokenizer)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model, tokenizer)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [10:09,  9.40it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/queries_embed.jsonl

Total embedding time: 620.42 seconds
Average time per query: 0.1083 sec


In [ ]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

torch.cuda.ipc_collect()

In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))


query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

Loading all embeddings from: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/queries_embed.jsonl
Performing batch search for 5731 queries...

================ RETRIEVAL STATS ================
Total queries           : 5731
Total retrieval time    : 4.7179 sec
Avg time per query      : 0.000823 sec
Throughput             : 1214.73 queries/sec

Batch retrieval complete.


#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    tokenizer,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model,
        tokenizer=tokenizer
    )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            tokenizer=tokenizer,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")



Running indexing experiment 1/3...
batch 1 | size=128 | time=0.6105s
batch 2 | size=128 | time=0.7091s
batch 3 | size=128 | time=0.7245s
batch 4 | size=128 | time=0.8603s
batch 5 | size=128 | time=0.7523s
batch 6 | size=128 | time=0.8443s
batch 7 | size=128 | time=0.8288s
batch 8 | size=128 | time=0.7427s
batch 9 | size=128 | time=0.8414s
batch 10 | size=128 | time=0.9084s
batch 11 | size=128 | time=0.8620s
batch 12 | size=128 | time=0.9602s
batch 13 | size=128 | time=0.9736s
batch 14 | size=128 | time=1.1999s
batch 15 | size=128 | time=1.0833s
batch 16 | size=128 | time=0.9733s
batch 17 | size=128 | time=1.1291s
batch 18 | size=128 | time=1.1298s
batch 19 | size=128 | time=1.1289s
batch 20 | size=128 | time=1.1462s
batch 21 | size=128 | time=1.0324s
batch 22 | size=128 | time=1.0359s
batch 23 | size=128 | time=1.2056s
batch 24 | size=128 | time=1.1318s
batch 25 | size=128 | time=1.2481s
batch 26 | size=128 | time=1.2027s
batch 27 | size=128 | time=1.0663s
batch 28 | size=128 | time=1

In [ ]:
import time
import numpy as np


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embed_query(
            queries[0],
        )
        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embed_query(
                query,
            )

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)


    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }

#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 3/3 [22:51<00:00, 457.09s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 3
Total time              : 457.08 sec
Avg latency per query   : 79.76 ms
Throughput              : 12.54 queries/sec


In [ ]:
import os

def get_index_size(index_path):
    """
    Returns FAISS index size in MB/GB.
    """
    size_bytes = os.path.getsize(index_path)

    return {
        "size_bytes": size_bytes,
        "size_MB": size_bytes / (1024 ** 2),
        "size_GB": size_bytes / (1024 ** 3)
    }

index_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bge-m3/sa_faiss.index"

index_size = get_index_size(index_file)

print(f"Index size: {index_size['size_MB']:.2f} MB")
print(f"Index size: {index_size['size_GB']:.3f} GB")

Index size: 87.38 MB
Index size: 0.085 GB


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2630,0.0966,NaN,NaN
1,10,0.3192,0.0650,0.2088,0.2293
2,100,0.5155,0.0118,NaN,NaN


### b. e5-large

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('intfloat/multilingual-e5-large')

In [ ]:
def embed_texts(batch_texts, model):
    """
    BGE-M3 HF embedding (dense only)
    """
    return model.encode(batch_texts)

def build_faiss_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    dim = np.array(embeddings).shape[1]
    index = faiss.IndexFlatIP(dim)
    index = faiss.IndexIDMap(index)
    index.add_with_ids(embeddings, ids)

    # --- SAVE ARTIFACTS ---
    if save_path:
        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

Loading documents...
Total documents: 22326
batch 1 | size=128 | time=0.8386s
batch 2 | size=128 | time=0.9213s
batch 3 | size=128 | time=1.0383s
batch 4 | size=128 | time=1.0420s
batch 5 | size=128 | time=1.1159s
batch 6 | size=128 | time=1.1429s
batch 7 | size=128 | time=1.1948s
batch 8 | size=128 | time=1.4711s
batch 9 | size=128 | time=1.8364s
batch 10 | size=128 | time=2.1579s
batch 11 | size=128 | time=2.2954s
batch 12 | size=128 | time=1.6781s
batch 13 | size=128 | time=1.6832s
batch 14 | size=128 | time=1.4776s
batch 15 | size=128 | time=1.4434s
batch 16 | size=128 | time=1.4011s
batch 17 | size=128 | time=1.4410s
batch 18 | size=128 | time=1.4023s
batch 19 | size=128 | time=1.4672s
batch 20 | size=128 | time=1.4502s
batch 21 | size=128 | time=1.4685s
batch 22 | size=128 | time=1.4430s
batch 23 | size=128 | time=1.3688s
batch 24 | size=128 | time=1.4559s
batch 25 | size=128 | time=1.4369s
batch 26 | size=128 | time=1.4821s
batch 27 | size=128 | time=1.3772s
batch 28 | size=128 

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()   #

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [06:31, 14.65it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/queries_embed.jsonl

Total embedding time: 398.23 seconds
Average time per query: 0.0695 sec


In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))


query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

Loading all embeddings from: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/queries_embed.jsonl
Performing batch search for 5731 queries...

================ RETRIEVAL STATS ================
Total queries           : 5731
Total retrieval time    : 4.9031 sec
Avg time per query      : 0.000856 sec
Throughput             : 1168.85 queries/sec

Batch retrieval complete.


#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model
    )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")



Running indexing experiment 1/3...
batch 1 | size=128 | time=0.5922s
batch 2 | size=128 | time=0.6868s
batch 3 | size=128 | time=0.7700s
batch 4 | size=128 | time=0.8075s
batch 5 | size=128 | time=0.8073s
batch 6 | size=128 | time=0.8514s
batch 7 | size=128 | time=0.8597s
batch 8 | size=128 | time=0.8110s
batch 9 | size=128 | time=0.8741s
batch 10 | size=128 | time=0.9184s
batch 11 | size=128 | time=0.9925s
batch 12 | size=128 | time=1.0309s
batch 13 | size=128 | time=1.0164s
batch 14 | size=128 | time=1.0708s
batch 15 | size=128 | time=1.0468s
batch 16 | size=128 | time=1.0686s
batch 17 | size=128 | time=1.0378s
batch 18 | size=128 | time=1.1299s
batch 19 | size=128 | time=1.1950s
batch 20 | size=128 | time=1.1967s
batch 21 | size=128 | time=1.1466s
batch 22 | size=128 | time=1.1630s
batch 23 | size=128 | time=1.2201s
batch 24 | size=128 | time=1.2464s
batch 25 | size=128 | time=1.2768s
batch 26 | size=128 | time=1.2226s
batch 27 | size=128 | time=1.2987s
batch 28 | size=128 | time=1

In [ ]:
import time
import numpy as np


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embedding_model.encode(
            [queries[0]],
            convert_to_numpy=True
        ).astype("float32")

        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embedding_model.encode(
                [query],
                convert_to_numpy=True
            ).astype("float32")

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)


    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }

#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=3
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 3/3 [24:28<00:00, 489.38s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 3
Total time              : 489.37 sec
Avg latency per query   : 85.39 ms
Throughput              : 11.71 queries/sec


In [ ]:
import os

def get_index_size(index_path):
    """
    Returns FAISS index size in MB/GB.
    """
    size_bytes = os.path.getsize(index_path)

    return {
        "size_bytes": size_bytes,
        "size_MB": size_bytes / (1024 ** 2),
        "size_GB": size_bytes / (1024 ** 3)
    }

index_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/e5-large/sa_faiss.index"

index_size = get_index_size(index_file)

print(f"Index size: {index_size['size_MB']:.2f} MB")
print(f"Index size: {index_size['size_GB']:.3f} GB")

Index size: 87.38 MB
Index size: 0.085 GB


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2358,0.0888,NaN,NaN
1,10,0.2825,0.0588,0.1873,0.2039
2,100,0.4588,0.0107,NaN,NaN


### c. Jina

In [ ]:
!pip install "transformers<5.0.0" "accelerate<1.0.0"

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)

In [ ]:
def embed_texts(batch_texts, model):
    """
    BGE-M3 HF embedding (dense only)
    """
    return model.encode(batch_texts)

def build_faiss_pipeline(
    texts,
    ids,
    model,
    batch_size=32,
    save_path=None
):
    all_embeddings = []
    total_time = 0.0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        start = time.time()

        emb = embed_texts(batch, model)

        end = time.time()
        total_time += (end - start)

        print(f"batch {i//batch_size + 1} | size={len(batch)} | time={end-start:.4f}s")

        all_embeddings.append(emb)

    embeddings = np.concatenate(all_embeddings, axis=0).astype("float32")

    dim = np.array(embeddings).shape[1]
    index = faiss.IndexFlatIP(dim)
    index = faiss.IndexIDMap(index)
    index.add_with_ids(embeddings, ids)

    # --- SAVE ARTIFACTS ---
    if save_path:
        os.makedirs(save_path, exist_ok=True)

        # 1. save faiss index
        faiss.write_index(index, os.path.join(save_path, "sa_faiss.index"))

        # 2. save embeddings
        np.save(os.path.join(save_path, "sa_embeddings.npy"), embeddings)

    print("\n--- summary ---")
    print(f"total embedding time: {total_time:.2f}s")
    print(f"total docs: {len(ids)}")
    print(f"dimension: {dim}")

    return index,embeddings

In [ ]:
#embed chunks
# CONFIG
input_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"
output_dir = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/"

BATCH_SIZE = 128

os.makedirs(output_dir, exist_ok=True)

# LOAD DOCUMENTS
data = []

print("Loading documents...")

with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        text = item["article_heading"] + ": " + item["text"]

        data.append({
            "id": item["sub_article_id"],
            "text": text,
            "len": len(text)
        })

# sort for batching efficiency
data.sort(key=lambda x: x["len"])

texts = [d["text"] for d in data]
ids = np.array([d["id"] for d in data], dtype=np.int64)

num_docs = len(texts)

print(f"Total documents: {num_docs}")

# time computation
start_time = time.time()

index, embeddings = build_faiss_pipeline(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path=output_dir
)

end_time = time.time()

# METRICS
total_time = end_time - start_time

avg_time_per_doc = total_time / num_docs
throughput = num_docs / total_time

# REPORT
print("\n================ INDEXING STATS ================")
print(f"Total documents embedded : {num_docs}")
print(f"Total indexing time      : {total_time:.2f} sec")
print(f"Average time per doc     : {avg_time_per_doc:.6f} sec")
print(f"Throughput              : {throughput:.2f} docs/sec")
print("=================================================\n")

In [ ]:
#embed queries
BATCH_SIZE = 128

query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/queries_embed.jsonl"

os.makedirs(os.path.dirname(query_output_file), exist_ok=True)

batch_items = []
batch_texts = []

start_time = time.time()   #

with open(query_output_file, "w", encoding="utf-8") as out_f:
    with open(query_file, "r", encoding="utf-8") as in_f:

        for line in tqdm(in_f, desc="Embedding queries"):

            item = json.loads(line)
            text = item.get("query_text")

            if not text:
                continue

            batch_items.append(item)
            batch_texts.append(text)

            # ---- RUN BATCH ----
            if len(batch_texts) == BATCH_SIZE:

                embeddings = embed_texts(batch_texts, model)

                for it, emb in zip(batch_items, embeddings):
                    to_write = {
                        "query_id": it["query_id"],
                        "query_text": it["query_text"],
                        "embedding": emb.tolist(),
                        "gold": it["relevant_subarticles"]
                    }
                    out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

                batch_items = []
                batch_texts = []

        # ---- FLUSH REMAINING ----
        if batch_texts:
            embeddings = embed_texts(batch_texts, model)

            for it, emb in zip(batch_items, embeddings):
                to_write = {
                    "query_id": it["query_id"],
                    "query_text": it["query_text"],
                    "embedding": emb.tolist(),
                    "gold": it["relevant_subarticles"]
                }
                out_f.write(json.dumps(to_write, ensure_ascii=False) + "\n")

end_time = time.time()   # END TIMER

print("Done! Query embeddings written to:", query_output_file)

# FINAL STATS
total_time = end_time - start_time
print(f"\nTotal embedding time: {total_time:.2f} seconds")
print(f"Average time per query: {total_time / sum(1 for _ in open(query_file)):.4f} sec")

Embedding queries: 5731it [11:31,  8.29it/s]


Done! Query embeddings written to: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/citation_removed_queries_embed.jsonl

Total embedding time: 703.55 seconds
Average time per query: 0.1228 sec


In [ ]:
#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))

query_output_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/queries_embed.jsonl"

all_query_embeddings = []
all_gold_labels = []

print(f"Loading all embeddings from: {query_output_file}")

with open(query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_query_embeddings.append(item["embedding"])
        all_gold_labels.append(item["gold"])

# Convert to numpy matrix
query_matrix = np.array(all_query_embeddings).astype("float32")

print(f"Performing batch search for {len(query_matrix)} queries...")

# START TIMER (RETRIEVAL TIME ONLY)
start_time = time.time()

D, I = index.search(query_matrix, 100)

end_time = time.time()

# RETRIEVAL TIME METRICS
retrieval_time = end_time - start_time
num_queries = len(query_matrix)

print("\n================ RETRIEVAL STATS ================")
print(f"Total queries           : {num_queries}")
print(f"Total retrieval time    : {retrieval_time:.4f} sec")
print(f"Avg time per query      : {retrieval_time / num_queries:.6f} sec")
print(f"Throughput             : {num_queries / retrieval_time:.2f} queries/sec")
print("=================================================\n")

# FORMAT RESULTS FOR EVALUATION
all_results_for_metrics = []

for row_idx in range(len(I)):
    retrieved_ids = I[row_idx].tolist()
    all_results_for_metrics.append((retrieved_ids, all_gold_labels[row_idx]))

print("Batch retrieval complete.")

Loading all embeddings from: /content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/queries_embed.jsonl
Performing batch search for 5731 queries...

================ RETRIEVAL STATS ================
Total queries           : 5731
Total retrieval time    : 3.7192 sec
Avg time per query      : 0.000649 sec
Throughput             : 1540.93 queries/sec

Batch retrieval complete.


#### Efficiency

In [ ]:
def measure_indexing_time(
    texts,
    ids,
    model,
    batch_size,
    save_path="/content/",
    runs=3
):
    """
    Measure dense indexing time over multiple runs.
    """

    num_docs = len(texts)

    # Warmup
    _ = embed_texts(
        texts[:min(batch_size, num_docs)],
        model
      )

    times = []

    for run in range(runs):
        print(f"\nRunning indexing experiment {run+1}/{runs}...")

        start_time = time.perf_counter()

        index, embeddings = build_faiss_pipeline(
            texts=texts,
            ids=ids,
            model=model,
            batch_size=batch_size,
            save_path=save_path
        )

        end_time = time.perf_counter()

        elapsed = end_time - start_time
        times.append(elapsed)

        print(f"Run {run+1} time: {elapsed:.2f} sec")


    avg_indexing_time = np.mean(times)

    results = {
        "num_documents": num_docs,
        "runs": runs,
        "all_times": times,
        "avg_indexing_time_sec": avg_indexing_time,
        "avg_time_per_document_sec": avg_indexing_time / num_docs,
        "throughput_docs_per_sec": num_docs / avg_indexing_time
    }

    return results

indexing_stats = measure_indexing_time(
    texts=texts,
    ids=ids,
    model=model,
    batch_size=BATCH_SIZE,
    save_path="/content/",
    runs=3
)

print("\n============== INDEXING STATS ==============")
print(f"Documents embedded      : {indexing_stats['num_documents']}")
print(f"Runs                    : {indexing_stats['runs']}")
print(f"Individual times        : {indexing_stats['all_times']}")
print(f"Average indexing time   : {indexing_stats['avg_indexing_time_sec']:.2f} sec")
print(f"Avg time per document   : {indexing_stats['avg_time_per_document_sec']:.6f} sec")
print(f"Throughput              : {indexing_stats['throughput_docs_per_sec']:.2f} docs/sec")
print("============================================")


In [ ]:
import time
import numpy as np


def measure_dense_retrieval_latency(
    index,
    queries,
    embedding_model,
    top_k=100,
    warmup=True,
    runs=3
):
    """
    Measure end-to-end dense retrieval latency:
    text query -> embedding -> FAISS search
    """

    num_queries = len(queries)

    # Warm-up
    if warmup:
        emb = embedding_model.encode(
            [queries[0]],
            convert_to_numpy=True
        ).astype("float32")

        index.search(emb, top_k)

    times = []

    for _ in tqdm(range(runs)):

        start = time.perf_counter()

        for query in queries:

            # Query embedding
            query_embedding = embedding_model.encode(
                [query],
                convert_to_numpy=True
            ).astype("float32")

            # Retrieval
            index.search(query_embedding, top_k)

        end = time.perf_counter()

        times.append(end - start)


    avg_time = np.mean(times)

    return {
        "num_queries": num_queries,
        "top_k": top_k,
        "runs": runs,
        "total_time_sec": avg_time,
        "avg_latency_per_query_ms": (avg_time / num_queries) * 1000,
        "queries_per_second": num_queries / avg_time,
        "all_runs": times
    }

#load and run experiments
index_path="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/"
index = faiss.read_index(os.path.join(index_path, "sa_faiss.index"))
query_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"
query_df=pd.read_json(query_file,lines=True)
batch_texts=query_df['query_text'].tolist()

results=measure_dense_retrieval_latency(
    index=index,
    queries=batch_texts,
    embedding_model=model,
    top_k=100,
    warmup=True,
    runs=1
)

print("\n============== RETRIEVAL STATS ==============")
print(f"Total queries           : {results['num_queries']}")
print(f"Top-k                   : {results['top_k']}")
print(f"Runs                    : {results['runs']}")
print(f"Total time              : {results['total_time_sec']:.2f} sec")
print(f"Avg latency per query   : {results['avg_latency_per_query_ms']:.2f} ms")
print(f"Throughput              : {results['queries_per_second']:.2f} queries/sec")
print("============================================")

100%|██████████| 1/1 [09:18<00:00, 558.45s/it]


============== RETRIEVAL STATS ==============
Total queries           : 5731
Top-k                   : 100
Runs                    : 1
Total time              : 558.45 sec
Avg latency per query   : 97.44 ms
Throughput              : 10.26 queries/sec


In [ ]:
import os

def get_index_size(index_path):
    """
    Returns FAISS index size in MB/GB.
    """
    size_bytes = os.path.getsize(index_path)

    return {
        "size_bytes": size_bytes,
        "size_MB": size_bytes / (1024 ** 2),
        "size_GB": size_bytes / (1024 ** 3)
    }

index_file = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/sa_faiss.index"

index_size = get_index_size(index_file)

print(f"Index size: {index_size['size_MB']:.2f} MB")
print(f"Index size: {index_size['size_GB']:.3f} GB")

Index size: 87.38 MB
Index size: 0.085 GB


#### Effectiveness

In [ ]:
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions = []
    recalls = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        p = hits / k
        r = hits / len(gold) if len(gold) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
    avg_p = np.mean(precisions)
    avg_r = np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        ideal_rels = [1] * min(len(gold), k)
        idcg = dcg(ideal_rels)
        rels = [1 if p in gold else 0 for p in preds]
        dcg_val = dcg(rels)
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.2590,0.0964,NaN,NaN
1,10,0.3262,0.0672,0.2024,0.2263
2,100,0.5431,0.0126,NaN,NaN



# 3.Sparse

### a. BM25

In [ ]:
import json
import pickle
import os
import time
import numpy as np
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# 1. IMPORTS + TOKENIZER + STOPWORDS
import json, math, re
from collections import Counter, defaultdict
import numpy as np
import scipy.sparse as sp
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x

# Devanagari block + word chars, lowercased
TOKEN_RE = re.compile(r"[ऀ-ॣ०-ॿ\w]+", flags=re.UNICODE)
STOPWORDS_PATH = "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/bm25/stopwords_ne.json"

try:
    with open(STOPWORDS_PATH, encoding="utf-8") as f:
        STOPWORDS = frozenset(json.load(f)["stopwords"])
    print(f"Loaded {len(STOPWORDS)} stopwords")
except (FileNotFoundError, TypeError):
    STOPWORDS = frozenset()
    print("No stopwords file found -> running without stopword removal")

def tokenize(text, stopwords=None):
    if stopwords is None:
        stopwords = STOPWORDS
    toks = TOKEN_RE.findall(text.lower())
    if stopwords:
        toks = [t for t in toks if t not in stopwords]
    return toks


Loaded 304 stopwords


In [ ]:
#FAST BM25
# Precompute every BM25 weight ONCE into a sparse (vocab x docs) matrix.
# A query then becomes one sparse matrix-multiply -> scores for all docs.
class FastBM25:
    def __init__(self, k1=1.5, b=0.75, stopwords=frozenset()):
        self.k1, self.b, self.stopwords = k1, b, stopwords
        self.doc_ids = []
        self._vocab = {}
        self._weights = None      # sparse (vocab_size x num_docs)
    def get_index_size(self):
      size_bytes = (
          self._weights.data.nbytes +
          self._weights.indices.nbytes +
          self._weights.indptr.nbytes
      )

      return size_bytes / (1024 ** 2)  # MB

    def build(self, doc_ids, doc_texts):
        self.doc_ids = list(doc_ids)
        raw = defaultdict(list)
        doc_freq = Counter()
        doc_lengths = []
        for text in tqdm(doc_texts, desc="indexing"):
            tc = Counter(tokenize(text, self.stopwords))
            doc_lengths.append(sum(tc.values()))
            doc_idx = len(doc_lengths) - 1
            for term, tf in tc.items():
                raw[term].append((doc_idx, tf))
                doc_freq[term] += 1

        n = len(self.doc_ids)
        dl = np.array(doc_lengths, dtype=np.float32)
        avgdl = float(dl.mean()) if n else 1.0
        norms = self.k1 * (1 - self.b + self.b * dl / (avgdl or 1.0))

        self._vocab = {t: i for i, t in enumerate(raw)}
        rows, cols, data = [], [], []
        for term, postings in raw.items():
            df = doc_freq[term]
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))
            ids = np.fromiter((p[0] for p in postings), np.int32, len(postings))
            tfs = np.fromiter((p[1] for p in postings), np.float32, len(postings))
            w = idf * (tfs * (self.k1 + 1)) / (tfs + norms[ids])
            rows.append(np.full(len(postings), self._vocab[term], np.int32))
            cols.append(ids)
            data.append(w.astype(np.float32))

        self._weights = sp.csr_matrix(
            (np.concatenate(data), (np.concatenate(rows), np.concatenate(cols))),
            shape=(len(self._vocab), n), dtype=np.float32,
        )
        return self

    def retrieve_batch(self, queries, top_k=100):
        n = len(self.doc_ids)
        if n == 0:
            return [[] for _ in queries]
        limit = min(top_k, n)

        # build a (num_queries x vocab) 0/1 indicator matrix
        q_rows, q_cols = [], []
        for qi, q in enumerate(queries):
            for term in set(tokenize(q, self.stopwords)):
                tid = self._vocab.get(term)
                if tid is not None:
                    q_rows.append(qi)
                    q_cols.append(tid)
        if not q_rows:
            return [[] for _ in queries]

        qm = sp.csr_matrix(
            (np.ones(len(q_rows), np.float32), (q_rows, q_cols)),
            shape=(len(queries), len(self._vocab)), dtype=np.float32,
        )
        scores = (qm @ self._weights).toarray()   # (num_queries x num_docs)

        out = []
        for row in scores:
            idx = np.argpartition(-row, limit - 1)[:limit]
            idx = idx[np.argsort(-row[idx])]
            out.append([(self.doc_ids[int(i)], float(row[i])) for i in idx if row[i] > 0])
        return out

    def retrieve(self, query, top_k=100):
        return self.retrieve_batch([query], top_k)[0]


In [ ]:
#DATA LOADERS  (subarticles corpus + queries with gold)
_SPACE_RE = re.compile(r"\s+")

def _norm(s):
    return _SPACE_RE.sub(" ", s or "").strip()

def load_documents(doc_path):
    """Corpus doc text = 'article_heading: text', id = sub_article_id (kept as-is)."""
    ids, texts, seen = [], [], set()
    with open(doc_path, encoding="utf-8") as f:
        for line in tqdm(f, desc="docs"):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            did = item["sub_article_id"]
            if did in seen:            # keep first occurrence, like the local pipeline
                continue
            seen.add(did)
            ids.append(did)
            texts.append(_norm(f"{item.get('article_heading', '')}: {item.get('text', '')}"))
    return ids, texts

def load_queries(query_path, query_type="uncited"):
    """Returns (qids, qtexts, golds). gold = relevant_subarticles (list[int]).

    query_type='uncited' removes the raw_citation substring from the query text so
    the answer isn't leaked; 'cited' leaves the query untouched.
    """
    qids, qtexts, golds = [], [], []
    with open(query_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            text = _norm(str(item.get("query_text", "")))
            gold = item.get("relevant_subarticles") or []
            if text and gold:          # skip queries with no text or no gold
                qids.append(str(item.get("query_id", "")))
                qtexts.append(text)
                golds.append(list(gold))
    return qids, qtexts, golds

In [ ]:
import time

# CONFIG
DOC_PATH   = "/content/drive/MyDrive/LegalNLPBenchmark Data/documents/chunks/subarticles.jsonl"

# INDEXING
doc_ids, doc_texts = load_documents(DOC_PATH)
print(f"{len(doc_ids)} documents")

start_index = time.time()

bm25 = FastBM25(stopwords=STOPWORDS).build(
    doc_ids,
    doc_texts
)

end_index = time.time()

indexing_time = end_index - start_index

print("\n========== INDEXING ==========")
print(f"Documents           : {len(doc_ids)}")
print(f"Indexing time       : {indexing_time:.4f} sec")
print(f"Avg/doc             : {indexing_time/len(doc_ids):.6f} sec")
print(f"Docs/sec            : {len(doc_ids)/indexing_time:.2f}")
print("================================\n")


In [ ]:
bm25.get_index_size()

6.003780364990234

In [ ]:
import time

TOP_K      = 100

all_query_results = {}


QUERY_PATH = f"/content/drive/MyDrive/LegalNLPBenchmark Data/queries/single-hop/queries.jsonl"


# LOAD QUERIES
qids, qtexts, golds = load_queries(QUERY_PATH)

print(f"Queries: {len(qids)}")


# RETRIEVAL
start_retrieval = time.time()

retrieved = bm25.retrieve_batch(
    qtexts,
    top_k=TOP_K
)

end_retrieval = time.time()

retrieval_time = end_retrieval - start_retrieval


print(f"Retrieval time: {retrieval_time:.4f} sec")
print(f"Avg/query: {retrieval_time/len(qids):.6f} sec")


# FORMAT FOR EVALUATION
all_results_for_metrics = [
    (
        [doc_id for doc_id, _score in preds],
        gold
    )
    for preds, gold in zip(retrieved, golds)
]

print("\nFinished all query spans")

Queries: 5731
Retrieval time: 13.6126 sec
Avg/query: 0.002375 sec

Finished all query spans


In [ ]:
#METRICS
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions, recalls = [], []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        precisions.append(hits / k)
        recalls.append(hits / len(gold) if len(gold) > 0 else 0.0)
    avg_p, avg_r = np.mean(precisions), np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        idcg = dcg([1] * min(len(gold), k))
        dcg_val = dcg([1 if p in gold else 0 for p in preds])
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)


In [ ]:

# Define K values for Recall only
ks = [5, 10,100]

eval_results = []

if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:

    # Compute fixed metrics once (K=10)
    mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
    ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

    for k in ks:
        rec = recall_at_k(all_results_for_metrics, k=k)
        f1 = f1_at_k(all_results_for_metrics, k=k)

        eval_results.append({
            "K": k,
            "Recall": round(rec, 4),
            "F1": round(f1, 4),

            # only filled at K=10, else blank/None
            "MRR@10": round(mrr_10, 4) if k == 10 else None,
            "NDCG@10": round(ndcg_10, 4) if k == 10 else None
        })

    df_metrics = pd.DataFrame(eval_results)
    display(df_metrics)

else:
    print("Error: 'all_results_for_metrics' not found. Please run the retrieval cell first.")

,K,Recall,F1,MRR@10,NDCG@10
0,5,0.3593,0.1284,NaN,NaN
1,10,0.4166,0.0824,0.2972,0.3203
2,100,0.5996,0.0134,NaN,NaN


# 4.Hybrid Model

### a.Bm25&bge

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import json
import numpy as np
from tqdm import tqdm

In [ ]:
def min_max_normalize(results):
    """
    results: [(doc_id, score), ...]
    """
    if not results:
        return {}

    scores = [score for _, score in results]
    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return {doc_id: 1.0 for doc_id, _ in results}

    return {
        doc_id: (score - min_score) / (max_score - min_score)
        for doc_id, score in results
    }


def score_fusion(bm25_candidates, bge_candidates, alpha=0.5):
    """
    alpha:
        1.0 -> only BM25
        0.0 -> only BGE
        0.5 -> equal weighting
    """

    bm25_norm = min_max_normalize(bm25_candidates)
    bge_norm = min_max_normalize(bge_candidates)

    fused_scores = {}

    # BM25 contribution
    for doc_id, score in bm25_norm.items():
        fused_scores[doc_id] = alpha * score

    # BGE contribution
    for doc_id, score in bge_norm.items():
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + (1-alpha) * score

    # Rank documents
    ranked = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked

In [ ]:
### jina query import
jina_embeddings=[]
texts=[]
golds=[]
jina_query_output_file="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/queries_embed.jsonl"
with open(jina_query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        texts.append(item["query_text"])
        jina_embeddings.append(item["embedding"])
        golds.append(item["gold"])

# Load BGE FAISS index
jina_index = faiss.read_index(
    "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/jina-v3/sa_faiss.index"
)

In [ ]:
# Load jina query embeddings
jina_embeddings = np.asarray(
    jina_embeddings,
    dtype=np.float32
)

TOP_K = 100

hybrid_candidates = []

# Retrieve candidates
for i, query in enumerate(tqdm(texts, desc="Hybrid retrieval")):
    # BM25
    bm_candidates = bm25.retrieve(query, TOP_K)

    # BGE
    D, I = jina_index.search(
        jina_embeddings[i:i+1],
        TOP_K
    )
    bge_candidates = list(zip(I[0].tolist(), D[0].tolist()))
    hybrid_candidates.append(
        {
            "query": query,
            "bm25": bm_candidates,
            "jina": bge_candidates,
            "gold": golds[i]
        }
    )

Hybrid retrieval: 100%|██████████| 5731/5731 [02:40<00:00, 35.76it/s]


#### Effectiveness

In [ ]:
#METRICS
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions, recalls = [], []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        precisions.append(hits / k)
        recalls.append(hits / len(gold) if len(gold) > 0 else 0.0)
    avg_p, avg_r = np.mean(precisions), np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        idcg = dcg([1] * min(len(gold), k))
        dcg_val = dcg([1 if p in gold else 0 for p in preds])
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

# # only evaluate at cutoffs we actually retrieved
# ks = [k for k in [5, 10, 100] if k <= TOP_K]

# if 'all_results_for_metrics' in globals() and len(all_results_for_metrics) > 0:
#     mrr_10 = mrr_at_k(all_results_for_metrics, k=10)
#     ndcg_10 = ndcg_at_k(all_results_for_metrics, k=10)

#     eval_results = []
#     for k in ks:
#         eval_results.append({
#             "K": k,
#             "Recall": round(recall_at_k(all_results_for_metrics, k=k), 4),
#             "F1": round(f1_at_k(all_results_for_metrics, k=k), 4),
#             "MRR@10": round(mrr_10, 4) if k == 10 else None,
#             "NDCG@10": round(ndcg_10, 4) if k == 10 else None,
#         })
#     df_metrics = pd.DataFrame(eval_results)
#     display(df_metrics)
# else:
#     print("Error: 'all_results_for_metrics' not found. Run the retrieval cell (5) first.")

In [ ]:
import pandas as pd

alpha_results = []

for alpha in [round(i * 0.1, 1) for i in range(1, 10)]:
    all_results_for_metrics = []

    for item in hybrid_candidates:
        fused_candidates = score_fusion(
            item["bm25"],
            item["jina"],
            alpha=alpha
        )

        all_results_for_metrics.append(
            (
                [doc_id for doc_id, _score in fused_candidates],
                item["gold"]
            )
        )

    # Recall
    alpha_results.append({
        "alpha": alpha,
        "Recall@5": round(recall_at_k(all_results_for_metrics, k=5), 4),
        "Recall@10": round(recall_at_k(all_results_for_metrics, k=10), 4),
        "Recall@100": round(recall_at_k(all_results_for_metrics, k=100), 4),
        "MRR@10": round(mrr_at_k(all_results_for_metrics, k=10), 4),
        "NDCG@10": round(ndcg_at_k(all_results_for_metrics, k=10), 4),
    })

df_alpha_recall = pd.DataFrame(alpha_results)

display(df_alpha_recall)

,alpha,Recall@5,Recall@10,Recall@100,MRR@10,NDCG@10
0,0.1,0.2794,0.3437,0.6237,0.2190,0.2432
1,0.2,0.3032,0.3633,0.6312,0.2402,0.2642
2,0.3,0.3248,0.3946,0.6346,0.2613,0.2877
3,0.4,0.3587,0.4323,0.6388,0.2832,0.3131
4,0.5,0.3875,0.4509,0.6409,0.3128,0.3396
5,0.6,0.3914,0.4529,0.6420,0.3171,0.3437
6,0.7,0.3869,0.4476,0.6427,0.3144,0.3406
7,0.8,0.3763,0.4409,0.6420,0.3083,0.3343
8,0.9,0.3653,0.4304,0.6401,0.3010,0.3262


### b.Bm25&acostillio

In [ ]:
def min_max_normalize(results):
    """
    results: [(doc_id, score), ...]
    """
    if not results:
        return {}

    scores = [score for _, score in results]
    min_score = min(scores)
    max_score = max(scores)

    if max_score == min_score:
        return {doc_id: 1.0 for doc_id, _ in results}

    return {
        doc_id: (score - min_score) / (max_score - min_score)
        for doc_id, score in results
    }


def score_fusion(bm25_candidates, bge_candidates, alpha=0.5):
    """
    alpha:
        1.0 -> only BM25
        0.0 -> only BGE
        0.5 -> equal weighting
    """

    bm25_norm = min_max_normalize(bm25_candidates)
    bge_norm = min_max_normalize(bge_candidates)

    fused_scores = {}

    # BM25 contribution
    for doc_id, score in bm25_norm.items():
        fused_scores[doc_id] = alpha * score

    # BGE contribution
    for doc_id, score in bge_norm.items():
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + (1-alpha) * score

    # Rank documents
    ranked = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked

In [ ]:
### bge query import
acos_embeddings=[]
texts=[]
golds=[]
ascos_query_output_file="/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/queries_embed.jsonl"
with open(ascos_query_output_file, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        texts.append(item["query_text"])
        acos_embeddings.append(item["embedding"])
        golds.append(item["gold"])

# Load BGE FAISS index
acos_index = faiss.read_index(
    "/content/drive/MyDrive/LegalNLPBenchmark Data/experiments/retrieval/acost-nepali/sa_faiss.index"
)

In [ ]:
# Load BGE query embeddings
acos_embeddings = np.asarray(
    acos_embeddings,
    dtype=np.float32
)

TOP_K = 100

hybrid_candidates = []

# Retrieve candidates
for i, query in enumerate(tqdm(texts, desc="Hybrid retrieval")):
    # BM25
    bm_candidates = bm25.retrieve(query, TOP_K)

    # BGE
    D, I = acos_index.search(
        acos_embeddings[i:i+1],
        TOP_K
    )
    acos_candidates = list(zip(I[0].tolist(), D[0].tolist()))
    hybrid_candidates.append(
        {
            "query": query,
            "bm25": bm_candidates,
            "acos": acos_candidates,
            "gold": golds[i]
        }
    )


Hybrid retrieval: 100%|██████████| 5731/5731 [02:16<00:00, 41.96it/s]


In [ ]:
#METRICS
import numpy as np
import ast
import pandas as pd

def parse_gold(gold):
    if isinstance(gold, str):
        return list(map(int, ast.literal_eval(gold)))
    return gold

def mrr_at_k(results, k):
    rr_sum = 0.0
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        rr = 0.0
        for rank, pid in enumerate(preds, start=1):
            if pid in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr
    return rr_sum / len(results)

def f1_at_k(results, k):
    precisions, recalls = [], []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        hits = len(set(preds).intersection(gold))
        precisions.append(hits / k)
        recalls.append(hits / len(gold) if len(gold) > 0 else 0.0)
    avg_p, avg_r = np.mean(precisions), np.mean(recalls)
    return 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0.0

def recall_at_k(results, k=10):
    recalls = []

    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]

        if len(gold) == 0:
            recalls.append(0.0)
            continue

        hits = len(set(preds).intersection(gold))
        recalls.append(hits / len(gold))

    return np.mean(recalls)

def dcg(rels):
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(rels))

def ndcg_at_k(results, k):
    scores = []
    for preds, gold in results:
        gold = set(parse_gold(gold))
        preds = preds[:k]
        idcg = dcg([1] * min(len(gold), k))
        dcg_val = dcg([1 if p in gold else 0 for p in preds])
        scores.append(dcg_val / idcg if idcg > 0 else 0.0)
    return np.mean(scores)

In [ ]:
import pandas as pd

alpha_results = []

for alpha in [round(i * 0.1, 1) for i in range(1, 10)]:
    all_results_for_metrics = []

    for item in hybrid_candidates:
        fused_candidates = score_fusion(
            item["bm25"],
            item["acos"],
            alpha=alpha
        )

        all_results_for_metrics.append(
            (
                [doc_id for doc_id, _score in fused_candidates],
                item["gold"]
            )
        )

    # Recall
    alpha_results.append({
        "alpha": alpha,
        "Recall@5": round(recall_at_k(all_results_for_metrics, k=5), 4),
        "Recall@10": round(recall_at_k(all_results_for_metrics, k=10), 4),
        "Recall@100": round(recall_at_k(all_results_for_metrics, k=100), 4),
        "MRR@10": round(mrr_at_k(all_results_for_metrics, k=10), 4),
        "NDCG@10": round(ndcg_at_k(all_results_for_metrics, k=10), 4),
    })

df_alpha_recall = pd.DataFrame(alpha_results)

display(df_alpha_recall)

,alpha,Recall@5,Recall@10,Recall@100,MRR@10,NDCG@10
0,0.1,0.2634,0.3272,0.6434,0.1993,0.2253
1,0.2,0.2859,0.3508,0.6512,0.2239,0.2492
2,0.3,0.3102,0.3760,0.6582,0.2494,0.2740
3,0.4,0.3426,0.4135,0.6625,0.2760,0.3024
4,0.5,0.3782,0.4391,0.6644,0.3080,0.3331
5,0.6,0.3837,0.4426,0.6638,0.3155,0.3398
6,0.7,0.3800,0.4403,0.6628,0.3138,0.3382
7,0.8,0.3728,0.4345,0.6592,0.3076,0.3320
8,0.9,0.3667,0.4279,0.6516,0.3019,0.3262
